# PSNI data extraction
Extracts the monthly PSNI security series (Feb 2015 – Dec 2025) from the
official accompanying spreadsheet:
*Police Recorded Security Situation Statistics*, finalised figures to
31 March 2026, published 14 May 2026, PSNI Statistics Branch.
Source: https://www.psni.police.uk/official-statistics/security-situation-statistics

On Colab: upload the .xls next to this notebook (or mount Drive and set PATH),
then run all cells. Output: `data/psni_monthly.csv`.
Sanity check: FY 2024/25 shootings must sum to 18 (finalised annual bulletin).

In [ ]:
from pathlib import Path
import os
import sys

# Run from the repository root or notebooks/; an explicit override is optional.
_candidate = Path(os.environ.get("NARRATIVE_REPO_DIR", Path.cwd())).resolve()
REPO_DIR = next((p for p in [_candidate, *_candidate.parents]
                 if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open the notebook inside the repository or set NARRATIVE_REPO_DIR.")
os.chdir(REPO_DIR)
DATA_DIR = REPO_DIR / "data"
FIGURES_DIR = REPO_DIR / "figures"
BASE = str(DATA_DIR)
IN_COLAB = "google.colab" in sys.modules
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
print(f"Repository: {REPO_DIR}")

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
print(f"Input folder: {DATA_DIR}")


In [ ]:
# Install xlrd in Google Colab if necessary:
# !pip install xlrd --quiet

from pathlib import Path

import pandas as pd
import xlrd

DATA_DIR = REPO_DIR / "data"
FIGURES_DIR = REPO_DIR / "figures"

INPUT_FILE = (
    DATA_DIR
    / "March 2026 Accompanying excel spreadsheet for Security website ONLINE 14.05.2026a2.xls"
)

OUTPUT_FILE = DATA_DIR / "psni_monthly.csv"

START_DATE = pd.Timestamp("2015-02-01")
END_DATE = pd.Timestamp("2025-12-01")


In [ ]:

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    available_files = "\n".join(
        f"  - {file.name}" for file in DATA_DIR.iterdir()
    )

    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_FILE}\n\n"
        f"Files currently available in {DATA_DIR}:\n"
        f"{available_files or '  No files found'}"
    )

print(f"Input file: {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")




In [ ]:

workbook = xlrd.open_workbook(str(INPUT_FILE))


def extract_monthly_series(
    workbook: xlrd.book.Book,
    sheet_name: str,
    column_mapping: dict[int, str],
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
) -> pd.DataFrame:
    """
    Extract monthly observations from an Excel worksheet.

    Parameters
    ----------
    workbook
        Open xlrd workbook.
    sheet_name
        Name of the worksheet to read.
    column_mapping
        Mapping between zero-based Excel column indices and output names.
        Example: {1: "shootings", 2: "bombings"}.
    start_date
        First month to retain.
    end_date
        Last month to retain.

    Returns
    -------
    pandas.DataFrame
        Monthly observations indexed by the first day of each month.
    """
    if sheet_name not in workbook.sheet_names():
        raise ValueError(
            f"Worksheet '{sheet_name}' was not found. "
            f"Available worksheets: {workbook.sheet_names()}"
        )

    sheet = workbook.sheet_by_name(sheet_name)
    records = []

    for row_index in range(sheet.nrows):
        first_cell = sheet.cell(row_index, 0)

        # Monthly rows contain a valid Excel date in the first column.
        # Rows such as "TOTAL 2025" are therefore ignored.
        if first_cell.ctype != xlrd.XL_CELL_DATE:
            continue

        excel_date = xlrd.xldate_as_datetime(
            first_cell.value,
            workbook.datemode,
        )

        record = {
            "month": pd.Timestamp(
                year=excel_date.year,
                month=excel_date.month,
                day=1,
            )
        }

        for column_index, output_name in column_mapping.items():
            cell = sheet.cell(row_index, column_index)

            if cell.ctype == xlrd.XL_CELL_NUMBER:
                record[output_name] = float(cell.value)
            else:
                record[output_name] = pd.NA

        records.append(record)

    if not records:
        raise ValueError(
            f"No monthly observations were found in worksheet "
            f"'{sheet_name}'."
        )

    dataframe = (
        pd.DataFrame(records)
        .set_index("month")
        .sort_index()
    )

    return dataframe.loc[start_date:end_date]

In [ ]:
incidents = extract_monthly_series(
    workbook=workbook,
    sheet_name="Security Related Incidents",
    column_mapping={
        1: "psni_shootings",
        2: "psni_bombings",
    },
    start_date=START_DATE,
    end_date=END_DATE,
)

attacks = extract_monthly_series(
    workbook=workbook,
    sheet_name="Paramilitary Style Attacks",
    column_mapping={
        1: "psni_para_shootings",
        4: "psni_para_assaults",
    },
    start_date=START_DATE,
    end_date=END_DATE,
)

arrests = extract_monthly_series(
    workbook=workbook,
    sheet_name="Terrorism Act arrests & charges",
    column_mapping={
        1: "psni_s41_arrests",
    },
    start_date=START_DATE,
    end_date=END_DATE,
)

In [ ]:
psni = incidents.join(
    [attacks, arrests],
    how="outer",
).sort_index()

psni.to_csv(
    OUTPUT_FILE,
    index=True,
    index_label="month",
)

print(
    f"\nDataset shape: {psni.shape}\n"
    f"Period: {psni.index.min().date()} "
    f"to {psni.index.max().date()}\n"
)

print("Missing values per column:")
print(psni.isna().sum())

print(f"\nCSV successfully saved to:\n{OUTPUT_FILE}")

In [ ]:

FY2024_25 = psni.loc["2024-04":"2025-03"]

expected_shootings = 18
observed_shootings = int(FY2024_25["psni_shootings"].sum())

assert observed_shootings == expected_shootings, (
    f"Sanity check failed: expected {expected_shootings} shootings "
    f"for FY2024/25, found {observed_shootings}."
)

print(f"✓ Sanity check passed: FY2024/25 shootings = {observed_shootings}")

# Save the processed dataset
psni.to_csv(OUTPUT_FILE, index=True, index_label="month")

print(f"Dataset saved to:\n{OUTPUT_FILE}")